# Milestone 2: Data Cleaning & Exploration

This notebook begins the data cleaning phase for the Job Market Intelligence Dashboard.

## Goals for this notebook

- Load the Kaggle job postings dataset
- Inspect the dataset structure and column names
- Identify data quality issues
- Prepare a roadmap for cleaning and transformation

**Note:** We are using the downloaded Kaggle dataset at `data/raw/job_postings_kaggle.csv`.


In [1]:
import ast
import re
import pandas as pd
from pathlib import Path

# File path for the Kaggle data
raw_data_path = Path('../data/raw/job_postings_kaggle.csv')

# Verify the file exists before loading
print('CSV path:', raw_data_path.resolve())
print('Exists:', raw_data_path.exists())


CSV path: C:\.projects\job-market-bi\data\raw\job_postings_kaggle.csv
Exists: True


In [2]:
# Load the first rows to inspect structure
raw_df = pd.read_csv(raw_data_path)

# Remove exported index columns that are not useful for analysis
raw_df = raw_df.drop(columns=[col for col in raw_df.columns if col.startswith('Unnamed')])

print('Shape:', raw_df.shape)
print('Columns:', list(raw_df.columns))
raw_df.head(10)


Shape: (61953, 26)
Columns: ['index', 'title', 'company_name', 'location', 'via', 'description', 'extensions', 'job_id', 'thumbnail', 'posted_at', 'schedule_type', 'work_from_home', 'salary', 'search_term', 'date_time', 'search_location', 'commute_time', 'salary_pay', 'salary_rate', 'salary_avg', 'salary_min', 'salary_max', 'salary_hourly', 'salary_yearly', 'salary_standardized', 'description_tokens']


,index,title,company_name,location,via,description,extensions,job_id,thumbnail,posted_at,...,commute_time,salary_pay,salary_rate,salary_avg,salary_min,salary_max,salary_hourly,salary_yearly,salary_standardized,description_tokens
0,0,Data Analyst,Meta,Anywhere,via LinkedIn,In the intersection of compliance and analytic...,"['15 hours ago', '101K–143K a year', 'Work fro...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,https://encrypted-tbn0.gstatic.com/images?q=tb...,15 hours ago,...,NaN,101K–143K,a year,122000.0,101000.0,143000.0,NaN,122000.0,122000.0,"['tableau', 'r', 'python', 'sql']"
1,1,Data Analyst,ATC,United States,via LinkedIn,Job Title: Entry Level Business Analyst / Prod...,"['12 hours ago', 'Full-time', 'Health insurance']",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,https://encrypted-tbn0.gstatic.com/images?q=tb...,12 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[]
2,2,Aeronautical Data Analyst,"Garmin International, Inc.","Olathe, KS",via Indeed,Overview:\n\nWe are seeking a full-time...\nAe...,"['18 hours ago', 'Full-time']",eyJqb2JfdGl0bGUiOiJBZXJvbmF1dGljYWwgRGF0YSBBbm...,NaN,18 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['sql']
3,3,Data Analyst - Consumer Goods - Contract to Hire,Upwork,Anywhere,via Upwork,Enthusiastic Data Analyst for processing sales...,"['12 hours ago', '15–25 an hour', 'Work from h...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QgLSBDb25zdW...,NaN,12 hours ago,...,NaN,15–25,an hour,20.0,15.0,25.0,20.0,NaN,41600.0,"['powerpoint', 'excel', 'power_bi']"
4,4,Data Analyst | Workforce Management,Krispy Kreme,United States,via LinkedIn,Overview of Position\n\nThis position will be ...,"['7 hours ago', '90K–110K a year', 'Contractor']",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QgfCBXb3JrZm...,https://encrypted-tbn0.gstatic.com/images?q=tb...,7 hours ago,...,NaN,90K–110K,a year,100000.0,90000.0,110000.0,NaN,100000.0,100000.0,"['powerpoint', 'excel', 'outlook', 'word']"
5,5,Data Analyst,Flint Hills Resources,"Wichita, KS",via Jora,Your Job\n\nFlint Hills Resources is seeking a...,"['20 hours ago', 'Full-time', 'Health insuranc...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,https://encrypted-tbn0.gstatic.com/images?q=tb...,20 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['power_bi', 'aws', 'excel', 'sql', 'mysql', '..."
6,6,Data Analyst (Model Validation),Swedbank,Anywhere,via JobTeaser,"Are you passionate about Credit Risk, Data Ana...","['17 hours ago', 'Work from home', 'Full-time'...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QgKE1vZGVsIF...,https://encrypted-tbn0.gstatic.com/images?q=tb...,17 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['python', 'sas', 'sql', 'spss']"
7,7,Data Analyst,AEG,United States,via IT Job Depot,"In order to be considered for this role, after...","['8 hours ago', 'Full-time']",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,https://encrypted-tbn0.gstatic.com/images?q=tb...,8 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['power_bi', 'tableau', 'jira', 'snowflake', '..."
8,8,IT Financial Data Analyst Lead,Progressive,Anywhere,via Jobgether,"This a Full Remote job, the offer is available...","['7 hours ago', 'Work from home', 'Full-time',...",eyJqb2JfdGl0bGUiOiJJVCBGaW5hbmNpYWwgRGF0YSBBbm...,https://encrypted-tbn0.gstatic.com/images?q=tb...,7 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[]
9,9,Data Analyst,Chloeta,"Oklahoma City, OK",via ZipRecruiter,Job Summary: The Data Analyst oversees data pr...,"['21 hours ago', 'Full-time', 'Health insuranc...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,NaN,21 hours ago,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['r', 'python']"


## Initial Data Exploration

Next we will:

- Check for missing values
- Inspect data types
- Review the first rows of the dataset
- Identify fields that need cleaning or parsing


In [3]:
# Basic data diagnostics
missing_counts = raw_df.isna().sum()
data_types = raw_df.dtypes

print('Missing values by column:')
print(missing_counts.sort_values(ascending=False).head(20))
print('\nData types:')
print(data_types)


Missing values by column:
commute_time           61953
salary_yearly          57884
salary_hourly          56053
salary_max             52441
salary_min             52441
salary_standardized    51865
salary_rate            51865
salary_avg             51865
salary_pay             51865
salary                 51865
work_from_home         33973
thumbnail              23759
schedule_type            246
posted_at                190
location                  37
via                        9
index                      0
search_term                0
search_location            0
date_time                  0
dtype: int64

Data types:
index                    int64
title                   object
company_name            object
location                object
via                     object
description             object
extensions              object
job_id                  object
thumbnail               object
posted_at               object
schedule_type           object
work_from_home          obj

## Cleaning Plan

Based on the dataset structure, the main cleaning tasks are:

- Normalize `company_name` text
- Clean `location` and create a consistent location field
- Convert `posted_at` and `date_time` to date/time values
- Normalize `schedule_type` and `work_from_home` into categories
- Parse the salary fields into consistent numeric values
- Extract skills and requirements from `description`
- Identify and handle duplicates


## Column-Level Exploration

We now inspect the most important columns that need cleaning and transformation:

- `company_name`
- `location`
- `schedule_type`
- `work_from_home`
- `salary`, `salary_min`, `salary_max`, `salary_yearly`
- `posted_at`, `date_time`
- `description`


In [4]:
# Inspect important columns for cleaning
columns_to_check = [
    'company_name', 'location', 'schedule_type', 'work_from_home',
    'salary', 'salary_min', 'salary_max', 'salary_yearly',
    'posted_at', 'date_time'
]

for col in columns_to_check:
    if col in raw_df.columns:
        print(f'Column: {col}')
        print(raw_df[col].dropna().astype(str).head(10).tolist())
        print(raw_df[col].value_counts(dropna=False).head(10))
        print('-' * 80)
    else:
        print(f'Column not found: {col}')
        print('-' * 80)


Column: company_name
['Meta', 'ATC', 'Garmin International, Inc.', 'Upwork', 'Krispy Kreme', 'Flint Hills Resources', 'Swedbank', 'AEG', 'Progressive', 'Chloeta']
company_name
Upwork                7533
Talentify.io          2118
Walmart               1829
vmysmartpros          1415
Dice                   862
EDWARD JONES           747
Corporate              612
Cox Communications     538
Insight Global         483
iSay                   353
Name: count, dtype: int64
--------------------------------------------------------------------------------
Column: location
[' Anywhere ', '  United States   ', '  Olathe, KS   ', ' Anywhere ', '  United States   ', '  Wichita, KS   ', ' Anywhere ', '  United States   ', ' Anywhere ', '  Oklahoma City, OK   ']
location
 Anywhere                  18067
  United States            10011
Anywhere                    9915
United States               5547
Denver, CO                  1062
  Oklahoma City, OK         1028
  Kansas City, MO            840
Ka

## Cleaning and Feature Engineering

The next cells apply cleaning rules to key fields and create derived features for analysis.

In [5]:
# Make a cleaned copy of the raw dataset
clean_df = raw_df.copy()

# Remove duplicate postings before feature engineering
rows_before_deduplication = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=['job_id'], keep='first')
rows_after_job_id_deduplication = len(clean_df)

clean_df['title_key'] = clean_df['title'].astype(str).str.strip().str.lower()
clean_df['company_key'] = clean_df['company_name'].astype(str).str.strip().str.lower()
clean_df['location_key'] = clean_df['location'].astype(str).str.strip().str.lower()
clean_df['date_time_key'] = clean_df['date_time'].astype(str).str.strip()

clean_df = clean_df.drop_duplicates(
    subset=['title_key', 'company_key', 'location_key', 'date_time_key'],
    keep='first'
)
clean_df = clean_df.drop(columns=['title_key', 'company_key', 'location_key', 'date_time_key'])
rows_after_deduplication = len(clean_df)

# Normalize text fields
def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = ' '.join(str(value).strip().split())
    return text if text else pd.NA

clean_df['title_clean'] = clean_df['title'].apply(clean_text)
clean_df['company_name_clean'] = clean_df['company_name'].apply(clean_text)
clean_df['location_clean'] = clean_df['location'].apply(clean_text)
clean_df['via_clean'] = clean_df['via'].apply(clean_text)
clean_df['posting_source_clean'] = clean_df['via_clean'].str.replace(r'^via\s+', '', case=False, regex=True)
clean_df['schedule_type_clean'] = clean_df['schedule_type'].apply(clean_text)
clean_df['search_term_clean'] = clean_df['search_term'].apply(clean_text)

# Normalize remote work
# The raw `work_from_home` field can contain booleans and strings.
# We convert both forms into consistent category labels.
clean_df['work_from_home_clean'] = clean_df['work_from_home'].replace({
    True: 'Remote',
    False: 'On-site',
    'True': 'Remote',
    'False': 'On-site'
})
clean_df['work_from_home_clean'] = clean_df['work_from_home_clean'].astype('string')
clean_df.loc[clean_df['work_from_home_clean'].isna(), 'work_from_home_clean'] = 'Unknown'

# Simplify schedule type
# This reduces compound values like 'Full-time and Part-time' into a single primary category.
def simplify_schedule(value):
    if pd.isna(value):
        return 'Unknown'
    text = str(value).lower()
    if 'full-time' in text:
        return 'Full-time'
    if 'part-time' in text:
        return 'Part-time'
    if 'contractor' in text:
        return 'Contractor'
    if 'intern' in text:
        return 'Internship'
    if 'temp' in text:
        return 'Temp'
    if 'volunteer' in text:
        return 'Volunteer'
    return 'Other'

clean_df['schedule_type_simplified'] = clean_df['schedule_type_clean'].apply(simplify_schedule)

# Remote work status
# Keep explicit remote signals separate from broad/unclear locations.
# `United States` is useful geography information, but it is not enough to call a job remote.
clean_df['is_remote_explicit'] = clean_df['work_from_home_clean'] == 'Remote'
clean_df['is_remote_location_inferred'] = clean_df['location_clean'].eq('Anywhere')
clean_df['is_broad_location'] = clean_df['location_clean'].isin(['Anywhere', 'United States'])
clean_df['is_remote'] = clean_df['is_remote_explicit'] | clean_df['is_remote_location_inferred']

def assign_remote_status(row):
    location = row['location_clean']
    if row['is_remote_explicit']:
        return 'Remote'
    if pd.isna(location):
        return 'Unknown'
    if location == 'Anywhere':
        return 'Likely Remote - Anywhere'
    if row['work_from_home_clean'] == 'On-site':
        return 'On-site/Hybrid'
    if location == 'United States':
        return 'Broad Location - Unclear'
    return 'Location-Based/Unclear'

clean_df['remote_status'] = clean_df.apply(assign_remote_status, axis=1)

# Parse dates
# posted_at is usually relative text like '15 hours ago', so estimate the posting date from date_time.
clean_df['date_time_parsed'] = pd.to_datetime(clean_df['date_time'], errors='coerce')

def parse_posting_age_days(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in ['just now', 'today']:
        return 0
    if text == 'yesterday':
        return 1
    match = re.search(r'(\d+)\+?\s*(minute|minutes|hour|hours|day|days|week|weeks|month|months|year|years)', text)
    if not match:
        return pd.NA
    amount = int(match.group(1))
    unit = match.group(2)
    if unit.startswith('minute') or unit.startswith('hour'):
        return 0
    if unit.startswith('day'):
        return amount
    if unit.startswith('week'):
        return amount * 7
    if unit.startswith('month'):
        return amount * 30
    if unit.startswith('year'):
        return amount * 365
    return pd.NA

clean_df['posting_age_days'] = clean_df['posted_at'].apply(parse_posting_age_days).astype('Int64')
clean_df['estimated_posted_date'] = (
    clean_df['date_time_parsed'] - pd.to_timedelta(clean_df['posting_age_days'].fillna(0), unit='D')
).dt.date
clean_df.loc[clean_df['posting_age_days'].isna(), 'estimated_posted_date'] = pd.NaT
clean_df['posted_weekday'] = pd.to_datetime(clean_df['estimated_posted_date'], errors='coerce').dt.day_name()

# Parse location into components
# Split city/state/country where possible so location analysis is easier.
def parse_location(location):
    if pd.isna(location) or location in ['', 'nan']:
        return pd.Series({'location_city': pd.NA, 'location_state': pd.NA, 'location_country': pd.NA})
    if location in ['Anywhere', 'United States']:
        return pd.Series({'location_city': pd.NA, 'location_state': pd.NA, 'location_country': location})
    parts = [part.strip() for part in location.split(',')]
    if len(parts) == 2:
        return pd.Series({'location_city': parts[0], 'location_state': parts[1], 'location_country': 'United States'})
    if len(parts) == 3:
        return pd.Series({'location_city': parts[0], 'location_state': parts[1], 'location_country': parts[2]})
    return pd.Series({'location_city': location, 'location_state': pd.NA, 'location_country': pd.NA})

clean_df = pd.concat([clean_df, clean_df['location_clean'].apply(parse_location)], axis=1)

# Salary cleaning
# Keep raw salary text, but add BI-friendly period, annualized min/max/midpoint, and salary bands.
salary_numeric_columns = ['salary_avg', 'salary_min', 'salary_max', 'salary_hourly', 'salary_yearly', 'salary_standardized']
for col in salary_numeric_columns:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

clean_df['salary_clean'] = clean_df['salary'].apply(clean_text)
clean_df['salary_pay_clean'] = clean_df['salary_pay'].apply(clean_text)
clean_df['salary_rate_clean'] = clean_df['salary_rate'].apply(clean_text)

def categorize_salary_period(value):
    if pd.isna(value):
        return 'Unknown'
    text = str(value).lower()
    if 'hour' in text:
        return 'Hourly'
    if 'month' in text:
        return 'Monthly'
    if 'year' in text:
        return 'Yearly'
    return 'Unknown'

def annualize_salary_value(value, period):
    if pd.isna(value) or period == 'Unknown':
        return pd.NA
    if period == 'Hourly':
        return float(value) * 2080
    if period == 'Monthly':
        return float(value) * 12
    if period == 'Yearly':
        return float(value)
    return pd.NA

def assign_salary_band(value):
    if pd.isna(value):
        return 'No Salary Listed'
    if value < 50000:
        return '<50K'
    if value < 75000:
        return '50K-75K'
    if value < 100000:
        return '75K-100K'
    if value < 125000:
        return '100K-125K'
    if value < 150000:
        return '125K-150K'
    return '150K+'

clean_df['salary_period'] = clean_df['salary_rate_clean'].apply(categorize_salary_period)
clean_df['salary_min_annual'] = [annualize_salary_value(value, period) for value, period in zip(clean_df['salary_min'], clean_df['salary_period'])]
clean_df['salary_max_annual'] = [annualize_salary_value(value, period) for value, period in zip(clean_df['salary_max'], clean_df['salary_period'])]
clean_df['salary_midpoint_annual'] = clean_df['salary_standardized']
missing_midpoint = clean_df['salary_midpoint_annual'].isna() & clean_df['salary_min_annual'].notna() & clean_df['salary_max_annual'].notna()
clean_df.loc[missing_midpoint, 'salary_midpoint_annual'] = (
    clean_df.loc[missing_midpoint, 'salary_min_annual'] + clean_df.loc[missing_midpoint, 'salary_max_annual']
) / 2
clean_df['salary_midpoint_annual'] = pd.to_numeric(clean_df['salary_midpoint_annual'], errors='coerce').round(2)
clean_df['salary_band'] = clean_df['salary_midpoint_annual'].apply(assign_salary_band)
clean_df['has_salary_info'] = clean_df['salary_midpoint_annual'].notna()

# Skill cleaning
# description_tokens already contains extracted skill-like tokens. Normalize them for BI and SQL analysis.
SKILL_ALIASES = {
    'power_bi': 'Power BI',
    'power bi': 'Power BI',
    'sql': 'SQL',
    't_sql': 'T-SQL',
    't-sql': 'T-SQL',
    't sql': 'T-SQL',
    'mysql': 'MySQL',
    'postgresql': 'PostgreSQL',
    'mongodb': 'MongoDB',
    'aws': 'AWS',
    'gcp': 'GCP',
    'azure': 'Azure',
    'python': 'Python',
    'r': 'R',
    'java': 'Java',
    'javascript': 'JavaScript',
    'typescript': 'TypeScript',
    'vba': 'VBA',
    'excel': 'Excel',
    'tableau': 'Tableau',
    'looker': 'Looker',
    'qlik': 'Qlik',
    'snowflake': 'Snowflake',
    'databricks': 'Databricks',
    'spark': 'Spark',
    'pandas': 'Pandas',
    'numpy': 'NumPy',
    'sas': 'SAS',
    'spss': 'SPSS',
    'jira': 'Jira',
    'git': 'Git',
    'github': 'GitHub',
    'gitlab': 'GitLab',
    'docker': 'Docker',
    'kubernetes': 'Kubernetes',
    'powerpoint': 'PowerPoint',
    'word': 'Word',
    'outlook': 'Outlook',
    'bigquery': 'BigQuery',
    'alteryx': 'Alteryx',
}

SKILL_CATEGORY_LOOKUP = {
    'Python': 'Programming Language', 'R': 'Programming Language', 'Java': 'Programming Language',
    'JavaScript': 'Programming Language', 'TypeScript': 'Programming Language', 'VBA': 'Programming Language',
    'SQL': 'Database/Query Language', 'T-SQL': 'Database/Query Language', 'MySQL': 'Database',
    'PostgreSQL': 'Database', 'MongoDB': 'Database', 'BigQuery': 'Database', 'Snowflake': 'Data Platform',
    'Power BI': 'BI Tool', 'Tableau': 'BI Tool', 'Looker': 'BI Tool', 'Qlik': 'BI Tool',
    'Excel': 'Spreadsheet/Office', 'PowerPoint': 'Spreadsheet/Office', 'Word': 'Spreadsheet/Office', 'Outlook': 'Spreadsheet/Office',
    'AWS': 'Cloud Platform', 'Azure': 'Cloud Platform', 'GCP': 'Cloud Platform', 'Databricks': 'Data Platform',
    'Spark': 'Data Engineering', 'Pandas': 'Python Library', 'NumPy': 'Python Library',
    'SAS': 'Analytics Tool', 'SPSS': 'Analytics Tool', 'Alteryx': 'ETL/Analytics Tool',
    'Jira': 'Project/Product Tool', 'Git': 'Developer Tool', 'GitHub': 'Developer Tool', 'GitLab': 'Developer Tool',
    'Docker': 'Developer Tool', 'Kubernetes': 'Developer Tool',
}

def normalize_skill_token(token):
    if pd.isna(token):
        return None
    text = str(token).strip().lower().replace('-', '_')
    text = ' '.join(text.replace('_', ' ').split())
    if not text:
        return None
    return SKILL_ALIASES.get(text, text.title())

def parse_skill_tokens(value):
    if pd.isna(value):
        return []
    try:
        tokens = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        tokens = []
    if not isinstance(tokens, list):
        return []
    normalized_skills = []
    for token in tokens:
        skill = normalize_skill_token(token)
        if skill and skill not in normalized_skills:
            normalized_skills.append(skill)
    return normalized_skills

def categorize_skill(skill):
    return SKILL_CATEGORY_LOOKUP.get(skill, 'Other')

clean_df['skills_list'] = clean_df['description_tokens'].apply(parse_skill_tokens)
clean_df['skills_clean'] = clean_df['skills_list'].apply(lambda skills: ', '.join(skills))
clean_df['skills_count'] = clean_df['skills_list'].apply(len)
clean_df['has_skills'] = clean_df['skills_count'] > 0

for skill_name in ['SQL', 'Python', 'Excel', 'Tableau', 'Power BI']:
    column_name = 'has_skill_' + skill_name.lower().replace(' ', '_').replace('-', '_')
    clean_df[column_name] = clean_df['skills_list'].apply(lambda skills, target=skill_name: target in skills)

skill_bridge_df = clean_df[['job_id', 'skills_list']].explode('skills_list').dropna(subset=['skills_list']).copy()
skill_bridge_df = skill_bridge_df.rename(columns={'skills_list': 'skill_name'})
skill_bridge_df['skill_category'] = skill_bridge_df['skill_name'].apply(categorize_skill)
skill_bridge_df = skill_bridge_df.drop_duplicates(subset=['job_id', 'skill_name']).reset_index(drop=True)

skills_dim_df = skill_bridge_df[['skill_name', 'skill_category']].drop_duplicates().sort_values('skill_name').reset_index(drop=True)
skills_dim_df.insert(0, 'skill_id', range(1, len(skills_dim_df) + 1))

# Job title category and seniority cleaning
def categorize_job_title(title):
    if pd.isna(title):
        return 'Other'
    text = str(title).lower()
    if re.search(r'\b(manager|director|head|vp|vice president)\b', text) and re.search(r'\b(data|analytics|analyst|bi|business intelligence)\b', text):
        return 'Analytics Manager/Leader'
    if 'data engineer' in text or 'analytics engineer' in text:
        return 'Data Engineer'
    if 'data scientist' in text or 'machine learning' in text:
        return 'Data Scientist'
    if 'business intelligence' in text or re.search(r'\bbi\b', text):
        if re.search(r'\b(developer|engineer|architect)\b', text):
            return 'BI Developer/Engineer'
        if 'analyst' in text:
            return 'BI Analyst'
        return 'Business Intelligence'
    if 'business analyst' in text:
        return 'Business Analyst'
    if 'product analyst' in text:
        return 'Product Analyst'
    if 'operations analyst' in text or 'operational analyst' in text:
        return 'Operations Analyst'
    if 'financial analyst' in text or 'finance analyst' in text:
        return 'Financial Analyst'
    if 'data analyst' in text or re.search(r'\banalyst\b', text):
        return 'Data Analyst'
    return 'Other'

def extract_min_experience_years(text):
    if pd.isna(text):
        return pd.NA
    matches = re.findall(r'(\d+)\+?\s*(?:years?|yrs?)\s+(?:of\s+)?(?:professional\s+)?(?:work\s+)?experience', str(text).lower())
    if not matches:
        return pd.NA
    return min(int(match) for match in matches)

def assign_seniority_level(row):
    title = '' if pd.isna(row['title_clean']) else str(row['title_clean']).lower()
    years = row['experience_years_min']
    if re.search(r'\b(intern|internship|entry level|entry-level|junior|jr\.?|graduate|trainee)\b', title):
        return 'Entry-Level'
    if re.search(r'\b(manager|director|head|vp|vice president|executive)\b', title):
        return 'Manager/Executive'
    if re.search(r'\b(senior|sr\.?|lead|principal|staff)\b', title):
        return 'Senior'
    if pd.notna(years):
        if years <= 2:
            return 'Entry-Level'
        if years <= 6:
            return 'Mid-Level'
        return 'Senior'
    return 'Unknown'

clean_df['job_title_category'] = clean_df['title_clean'].apply(categorize_job_title)
clean_df['experience_years_min'] = clean_df['description'].apply(extract_min_experience_years).astype('Int64')
clean_df['seniority_level'] = clean_df.apply(assign_seniority_level, axis=1)

print('Rows before duplicate cleaning:', rows_before_deduplication)
print('Rows removed by job_id deduplication:', rows_before_deduplication - rows_after_job_id_deduplication)
print('Rows removed by business-key deduplication:', rows_after_job_id_deduplication - rows_after_deduplication)
print('Cleaned dataset shape:', clean_df.shape)
print(clean_df[['title_clean', 'job_title_category', 'seniority_level', 'company_name_clean', 'location_clean', 'remote_status', 'schedule_type_simplified', 'work_from_home_clean', 'is_remote', 'posting_age_days', 'estimated_posted_date', 'salary_period', 'salary_midpoint_annual', 'salary_band', 'skills_clean', 'skills_count', 'has_salary_info']].head(10).to_string(index=False))
print('Unique normalized skills:', len(skills_dim_df))
print('Job-skill assignments:', len(skill_bridge_df))

Rows before duplicate cleaning: 61953
Rows removed by job_id deduplication: 3178
Rows removed by business-key deduplication: 74
Cleaned dataset shape: (58701, 68)
                                     title_clean job_title_category seniority_level         company_name_clean    location_clean            remote_status schedule_type_simplified work_from_home_clean  is_remote  posting_age_days estimated_posted_date salary_period  salary_midpoint_annual      salary_band                                                   skills_clean  skills_count  has_salary_info
                                    Data Analyst       Data Analyst       Mid-Level                       Meta          Anywhere                   Remote                Full-time               Remote       True                 0            2023-08-04        Yearly                122000.0        100K-125K                                        Tableau, R, Python, SQL             4             True
                                    D

## Cleaned Field Summaries

Review the summary counts for the transformed fields.

In [6]:
summary_columns = [
    'job_title_category', 'seniority_level', 'schedule_type_simplified', 'work_from_home_clean', 'remote_status',
    'is_remote', 'is_broad_location', 'location_country', 'location_state',
    'posting_age_days', 'salary_period', 'salary_band', 'has_salary_info'
]

for col in summary_columns:
    print(f'=== {col} ===')
    print(clean_df[col].value_counts(dropna=False).head(20))
    print('-' * 80)

print('=== Top cleaned locations ===')
print(clean_df['location_clean'].value_counts(dropna=False).head(20))

=== job_title_category ===
job_title_category
Data Analyst                41117
Other                        7430
Data Scientist               2881
BI Analyst                   2177
Analytics Manager/Leader     1865
Data Engineer                1329
Business Analyst             1172
BI Developer/Engineer         236
Business Intelligence         183
Operations Analyst            132
Product Analyst               100
Financial Analyst              79
Name: count, dtype: int64
--------------------------------------------------------------------------------
=== seniority_level ===
seniority_level
Unknown              31903
Senior               13097
Mid-Level             6080
Entry-Level           5643
Manager/Executive     1978
Name: count, dtype: int64
--------------------------------------------------------------------------------
=== schedule_type_simplified ===
schedule_type_simplified
Full-time     44458
Contractor    12264
Part-time      1045
Internship      491
Unknown         244

## First 10 rows of the cleaned dataset

Show all columns so we can verify every cleaned and derived field.

In [7]:
# Create a comparison DataFrame for remote/work-from-home fields
compare_df = clean_df[[
    'work_from_home',
    'work_from_home_clean',
    'location',
    'location_clean',
    'is_remote'
]].copy()

# Compute explicit/inferred remote flags so this cell works even if previous cells were not rerun
compare_df['is_remote_explicit'] = compare_df['work_from_home_clean'] == 'Remote'
compare_df['is_remote_location_inferred'] = compare_df['location_clean'].isin(['Anywhere', 'United States'])

# Create a manual consistency check column
compare_df['remote_consistent'] = compare_df.apply(
    lambda row: 'Check' if (
        (str(row['work_from_home']).strip().lower() not in ['true', 'false', 'nan'] and pd.isna(row['work_from_home_clean']))
        or (row['work_from_home_clean'] == 'Remote' and not row['is_remote_explicit'])
        or (row['work_from_home_clean'] == 'On-site' and row['is_remote_explicit'])
    ) else 'OK',
    axis=1
)

with pd.option_context('display.max_columns', None, 'display.width', None):
    display(compare_df.head(10))

,work_from_home,work_from_home_clean,location,location_clean,is_remote,is_remote_explicit,is_remote_location_inferred,remote_consistent
0,True,Remote,Anywhere,Anywhere,True,True,True,OK
1,NaN,Unknown,United States,United States,False,False,True,OK
2,NaN,Unknown,"Olathe, KS","Olathe, KS",False,False,False,OK
3,True,Remote,Anywhere,Anywhere,True,True,True,OK
4,NaN,Unknown,United States,United States,False,False,True,OK
5,NaN,Unknown,"Wichita, KS","Wichita, KS",False,False,False,OK
6,True,Remote,Anywhere,Anywhere,True,True,True,OK
7,NaN,Unknown,United States,United States,False,False,True,OK
8,True,Remote,Anywhere,Anywhere,True,True,True,OK
9,NaN,Unknown,"Oklahoma City, OK","Oklahoma City, OK",False,False,False,OK


## Save Cleaned Dataset

Save the cleaned dataset so the next analysis steps can reuse it without re-running all cleaning logic.

In [8]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

job_postings_export_df = clean_df.drop(columns=['skills_list'], errors='ignore')
job_postings_export_df.to_csv(processed_dir / 'job_postings_clean.csv', index=False)
skills_dim_df.to_csv(processed_dir / 'skills_clean.csv', index=False)
skill_bridge_df.to_csv(processed_dir / 'job_skills_clean.csv', index=False)

print('Saved cleaned data to', (processed_dir / 'job_postings_clean.csv').resolve())
print('Saved skills dimension to', (processed_dir / 'skills_clean.csv').resolve())
print('Saved job-skill bridge to', (processed_dir / 'job_skills_clean.csv').resolve())

Saved cleaned data to C:\.projects\job-market-bi\data\processed\job_postings_clean.csv
Saved skills dimension to C:\.projects\job-market-bi\data\processed\skills_clean.csv
Saved job-skill bridge to C:\.projects\job-market-bi\data\processed\job_skills_clean.csv


In [9]:
# Day 2

In [10]:
clean_df

,index,title,company_name,location,via,description,extensions,job_id,thumbnail,posted_at,...,skills_count,has_skills,has_skill_sql,has_skill_python,has_skill_excel,has_skill_tableau,has_skill_power_bi,job_title_category,experience_years_min,seniority_level
0,0,Data Analyst,Meta,Anywhere,via LinkedIn,In the intersection of compliance and analytic...,"['15 hours ago', '101K–143K a year', 'Work fro...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,https://encrypted-tbn0.gstatic.com/images?q=tb...,15 hours ago,...,4,True,True,True,False,True,False,Data Analyst,3,Mid-Level
1,1,Data Analyst,ATC,United States,via LinkedIn,Job Title: Entry Level Business Analyst / Prod...,"['12 hours ago', 'Full-time', 'Health insurance']",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...,https://encrypted-tbn0.gstatic.com/images?q=tb...,12 hours ago,...,0,False,False,False,False,False,False,Data Analyst,<NA>,Unknown
2,2,Aeronautical Data Analyst,"Garmin International, Inc.","Olathe, KS",via Indeed,Overview:\n\nWe are seeking a full-time...\nAe...,"['18 hours ago', 'Full-time']",eyJqb2JfdGl0bGUiOiJBZXJvbmF1dGljYWwgRGF0YSBBbm...,NaN,18 hours ago,...,1,True,True,False,False,False,False,Data Analyst,<NA>,Unknown
3,3,Data Analyst - Consumer Goods - Contract to Hire,Upwork,Anywhere,via Upwork,Enthusiastic Data Analyst for processing sales...,"['12 hours ago', '15–25 an hour', 'Work from h...",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QgLSBDb25zdW...,NaN,12 hours ago,...,3,True,False,False,True,False,True,Data Analyst,<NA>,Unknown
4,4,Data Analyst | Workforce Management,Krispy Kreme,United States,via LinkedIn,Overview of Position\n\nThis position will be ...,"['7 hours ago', '90K–110K a year', 'Contractor']",eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QgfCBXb3JrZm...,https://encrypted-tbn0.gstatic.com/images?q=tb...,7 hours ago,...,4,True,False,False,True,False,False,Data Analyst,3,Mid-Level
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61948,955,Marketing Data & BI Analyst II,EDWARD JONES,"Houstonia, MO",via My ArkLaMiss Jobs,"At Edward Jones, we help clients achieve their...","['23 hours ago', '76,798–130,764 a year', 'Ful...",eyJqb2JfdGl0bGUiOiJNYXJrZXRpbmcgRGF0YSBcdTAwMj...,NaN,23 hours ago,...,7,True,True,True,True,True,True,BI Analyst,<NA>,Unknown
61949,956,Lead-Data Analyst,EDWARD JONES,"Marshfield, MO",via My ArkLaMiss Jobs,"At Edward Jones, we help clients achieve their...","['23 hours ago', '106,916–182,047 a year', 'Fu...",eyJqb2JfdGl0bGUiOiJMZWFkLURhdGEgQW5hbHlzdCIsIm...,NaN,23 hours ago,...,0,False,False,False,False,False,False,Data Analyst,<NA>,Senior
61950,957,Lead-Data Analyst,EDWARD JONES,"High Point, MO",via My ArkLaMiss Jobs,"At Edward Jones, we help clients achieve their...","['23 hours ago', '106,916–182,047 a year', 'Fu...",eyJqb2JfdGl0bGUiOiJMZWFkLURhdGEgQW5hbHlzdCIsIm...,NaN,23 hours ago,...,0,False,False,False,False,False,False,Data Analyst,<NA>,Senior
61951,958,Lead-Data Analyst,EDWARD JONES,"Calhoun, MO",via My ArkLaMiss Jobs,"At Edward Jones, we help clients achieve their...","['23 hours ago', '106,916–182,047 a year', 'Fu...",eyJqb2JfdGl0bGUiOiJMZWFkLURhdGEgQW5hbHlzdCIsIm...,NaN,23 hours ago,...,0,False,False,False,False,False,False,Data Analyst,<NA>,Senior


In [11]:
raw_df = clean_df

In [12]:
# Check and remove duplicate job postings

# 1. Exact duplicate check using job_id
print("Total rows before duplicate cleaning:", len(raw_df))

job_id_duplicates = raw_df.duplicated(subset=["job_id"]).sum()
print("Duplicate job_id rows:", job_id_duplicates)

# 2. Create normalized fields for broader duplicate detection
dedupe_df = raw_df.copy()

dedupe_df["title_key"] = dedupe_df["title"].astype(str).str.strip().str.lower()
dedupe_df["company_key"] = dedupe_df["company_name"].astype(str).str.strip().str.lower()
dedupe_df["location_key"] = dedupe_df["location"].astype(str).str.strip().str.lower()
dedupe_df["date_time_key"] = dedupe_df["date_time"].astype(str).str.strip()

# 3. Check likely duplicate postings
business_key = ["title_key", "company_key", "location_key", "date_time_key"]

business_duplicates = dedupe_df.duplicated(subset=business_key).sum()
print("Likely duplicate job rows:", business_duplicates)

# 4. Preview duplicates before removing them
duplicate_preview = dedupe_df[dedupe_df.duplicated(subset=business_key, keep=False)]

duplicate_preview[
    ["title", "company_name", "location", "via", "posted_at", "date_time", "job_id"]
].sort_values(["company_name", "title"]).head(20)

Total rows before duplicate cleaning: 58701
Duplicate job_id rows: 0


Likely duplicate job rows: 0


,title,company_name,location,via,posted_at,date_time,job_id


In [13]:
# Deduplication is now part of the main cleaning pipeline above.
# Keep this as a quick check instead of resetting clean_df.
print("Rows in raw data:", len(raw_df))
print("Rows in current cleaned data:", len(clean_df))
print("Rows removed during deduplication:", len(raw_df) - len(clean_df))

Rows in raw data: 58701
Rows in current cleaned data: 58701
Rows removed during deduplication: 0


In [14]:
# Confirm there are no duplicate business keys left in clean_df.
business_key_check = clean_df.assign(
    title_key=clean_df['title'].astype(str).str.strip().str.lower(),
    company_key=clean_df['company_name'].astype(str).str.strip().str.lower(),
    location_key=clean_df['location'].astype(str).str.strip().str.lower(),
    date_time_key=clean_df['date_time'].astype(str).str.strip()
)

remaining_business_duplicates = business_key_check.duplicated(
    subset=['title_key', 'company_key', 'location_key', 'date_time_key']
).sum()

print("Remaining likely business-key duplicates:", remaining_business_duplicates)

Remaining likely business-key duplicates: 0
